# TRINETRA Training Starter
Run this in Google Colab with **Runtime -> Change runtime type -> GPU**.

This notebook: clones your GitHub repo, installs ML dependencies, and gives you
a skeleton training loop to fill in during Phase 3 (CNN Classifier) and
Phase 5 (Probabilistic Ensemble Forecaster).

In [ ]:
# --- Step 1: clone your repo ---
# Replace with your actual GitHub URL once you've pushed the starter kit.
REPO_URL = "https://github.com/<your-username>/trinetra.git"
!git clone $REPO_URL
%cd trinetra

In [ ]:
# --- Step 2: install ML dependencies ---
!pip install -q -r ml/requirements.txt

In [ ]:
# --- Step 3: confirm GPU is actually available ---
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only -- go check Runtime settings")

## Step 4: mount Google Drive (for storing data + checkpoints)
Data files and trained model checkpoints are too large for Git -- keep them in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/trinetra_data'
CHECKPOINT_DIR = '/content/drive/MyDrive/trinetra_checkpoints'
import os
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## Phase 3 placeholder: CNN Classifier
Fill this in once Phase 2 (Data Acquisition) has produced labeled imagery.

In [ ]:
import torch.nn as nn
from torchvision import models

def build_classifier(num_classes=7):
    """IMD intensity grades: Depression, Deep Depression, Cyclonic Storm,
    Severe CS, Very Severe CS, Extremely Severe CS, Super Cyclonic Storm."""
    model = models.resnet18(weights='DEFAULT')
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

model = build_classifier()
model = model.cuda() if torch.cuda.is_available() else model
print(model.fc)

## Phase 5 placeholder: Monte Carlo Dropout Ensemble
This is the core of the Uncertainty Cone -- keep dropout active at inference time.

In [ ]:
def enable_mc_dropout(model):
    """Keeps dropout layers active during inference -- this is what turns a
    normal model into a Monte Carlo ensemble."""
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()
    return model

def mc_forward_passes(model, x, n_passes=50):
    """Runs N forward passes with dropout active, returns the full spread of
    predictions -- this spread is what becomes the uncertainty cone."""
    model.eval()
    enable_mc_dropout(model)
    preds = []
    with torch.no_grad():
        for _ in range(n_passes):
            preds.append(model(x))
    return torch.stack(preds)  # shape: [n_passes, batch, ...]